In [ ]:
!pip install langchain>=1.0.0
!pip install langchain-core>=1.0.0
!pip install langchain-openai
!pip install langchain langchain-anthropic langgraph
!pip install tiktoken
!pip install langchain-community jq
!pip install unstructured
!pip install chromadb
!pip install -U sentence-transformers
!pip install langchain-classic

  Using cached chromadb-1.5.7-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
  Using cached build-1.4.3-py3-none-any.whl.metadata (5.8 kB)
  Using cached pybase64-1.4.3-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (8.7 kB)
  Using cached onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.2 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.41.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-manylinux_2_34_x86_64.whl.metadata (10 kB)
  Using cached kubernetes-35.0.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from langchain_openai import ChatOpenAI
import os
import openai
from google.colab import userdata

# API 키 설정하기.
os.environ["OPENAI_API_KEY"] = userdata.get("OPEN_API")

# ChatOpenAI 모델 초기화하기.
llm = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature = 0.5
)

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, RecursiveJsonSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from collections import Counter
import json


In [ ]:
import numpy as np
from numpy import dot
from numpy.linalg import norm

def cos_sim(A,B):
  return dot(A,B) / (norm(A)*norm(B))

In [ ]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

embeddings_model = HuggingFaceBgeEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device' : 'cuda'},
    encode_kwargs={'normalize_embeddings':True},
    )

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
with open("drug_documents.json", "r") as json_file:
  json_data = json.load(json_file)


splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 100
)

data = [
    Document(
        page_content = item["page_content"],
        metadata=item["metadata"]
    )
    for item in json_data
]


docs = splitter.split_documents(data)

page_content='[어린이타이레놀산160밀리그램(아세트아미노펜) 설명서]

1.효능·효과
이 약은 감기로 인한 발열 및 동통(통증), 두통, 신경통, 근육통, 월경통, 염좌통(삔 통증), 치통, 관절통, 류마티양 동통(통증)에 사용합니다.

2.용법·용량
만 7~12세 소아는 1회 권장용량을 4~6시간마다 필요시 물 없이 혀에 직접 복용합니다. 이 약은 가능한 최단기간동안 최소 유효용량으로 복용하며, 1일 5회(75 mg/kg)를 초과하여 복용하지 않습니다. 몸무게를 아는 경우 몸무게에 따른 용량(10~15 mg/kg)으로 복용하는 것이 더 적절합니다. 연령대별 용량은 허가사항을 참고하십시오.' metadata={'drug_name': '어린이타이레놀산160밀리그램(아세트아미노펜)', 'ingredient': '아세트아미노펜과립', 'company': '켄뷰코리아판매유한회사', 'item_seq': '202005623'}
page_content='3.주의사항
매일 세잔 이상 정기적 음주자가 이 약 또는 다른 해열진통제를 복용할 때는 의사 또는 약사와 상의하십시오. 간손상을 일으킬 수 있습니다. 매우 드물게 치명적일 수 있는 급성전신발진고름물집증, 스티븐스-존슨증후군, 독성표피괴사용해와 같은 중대한 피부반응이 보고되었고 이 약 복용 후 피부발진 또는 다른 과민반응의 징후가 나타나는 경우 즉시 복용을 중단하십시오. 아세트아미노펜으로 일일 최대 용량(4,000 mg)을 초과하여 복용하지 마십시오. 간손상을 일으킬 수 있습니다. 아세트아미노펜을 포함하는 다른 제품과 함께 복용하지 마십시오.
이 약에 과민증 환자, 소화성궤양, 심한 혈액 이상, 심한 간장애, 심한 신장(콩팥)장애, 심한 심장기능저하 환자, 아스피린 천식(비스테로이드성 소염(항염)제에 의한 천식발작 유발) 또는 경험자는 이 약을 복용하지 마십시오. 이 약을 복용하기 전에 간장애 또는 경험자, 신장(콩팥)장애 또는 경험자, 소화성궤양 경험자, 혈액이상 또는 경험자, 출혈경향, 심장기능이상, 과민증 

In [ ]:
vectorstore = Chroma.from_documents(
    docs,
    embeddings_model,
    collection_name = 'medicines',
    persist_directory = './db/chromadb',
    collection_metadata = {'hnsw:space' : 'cosine'},
)

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k" : 5}
)

# 검색한 문서 결과를 한 개의 문단으로 합침.
def format_docs(document):
  result = []
  for doc in document:
    item_seq = doc.metadata.get("item_seq", "unknown")
    drug_name = doc.metadata.get("drug_name", "알 수 없음")
    ingredient = doc.metadata.get("ingredient", "")

    formatted = f"""
    [약 ID : {item_seq}]
    [약 이름]:
    {drug_name}

    [성분]:
    {ingredient}

    [설명]:
    {doc.page_content}

    """

    result.append(formatted.strip())
  return "\n\n---\n\n".join(result)


def extract_effects_with_llm(text):
  prompt = f"""
  다음 약 설명에서 '효능(증상)'만 추출하세요.

  조건:
  - 반드시 리스트 형태
  - 최대 5개
  - 일반화된 표현 사용

  텍스트 :
  {text}

  출력 예:
  ["두통", "발열", "근육통", "눈의 피로"]
  """

  result = llm.invoke(prompt)

  try:
    return json.loads(result.content)
  except:
    return []


def build_structured_doc(doc):
  effects = extract_effects_with_llm(doc.page_content)

  doc.page_content = f"""

  [약 이름]
  {doc.metadata.get("drug_name")}

  [효능]
  {effects}

  [설명]
  {doc.page_content}
  """

  return doc


def build_structured_query(query):
  symptoms = extract_effects_with_llm(query)

  return f"""
  [증상]
  {", ".join(symptoms)}

  [사용자 질문]
  {query}
  """

def filter_docs_with_llm(query, docs):
  doc_list = "\n".join([
      f"{i}. {d.metadata.get('drug_name')} - {d.page_content[:100]}"
      for i, d in enumerate(docs)
  ])

  prompt = f"""
  사용자의 증상 : {query}

  다음 약들 중에서 증상과 관련 있는 약만 선택하세요.

  약 목록:
  {doc_list}

  관련 있는 약의 번호만 JSON 리스트로 출력하세요.
  예 : [0,2]
  """

  result = llm.invoke(prompt)
  try:
    indices = json.loads(result.content)
    return [docs[i] for i in indices if i < len(docs)]
  except:
    return docs[:5]


def get_filtered_docs(x):
  if isinstance(x, dict):
    query = x.get("input", "")
  else:
    query = x

  structured_query = build_structured_query(query)

  docs = vectorstore.similarity_search(structured_query, k = 20)
  docs = filter_docs_with_llm(query, docs)

  if not docs:
    return ""


  source_file = docs[0].metadata.get("item_seq")

  main_docs = [
      d for d in docs
      if d.metadata.get("item_seq") == source_file
  ]

  other_drugs = []
  seen = set()

  for doc in docs:
    item_seq = doc.metadata.get("item_seq")
    drug_name = doc.metadata.get("drug_name")

    if item_seq != source_file and drug_name not in seen:
      seen.add(drug_name)
      other_drugs.append(doc)

    if len(other_drugs) >= 2:
      break

  main_context = format_docs(main_docs)
  similar_context = format_docs(other_drugs)

  return f"""
  다음은 사용 가능한 약 목록입니다.
  [대표 약]
  {main_context}

  [유사 효능 약]
  {similar_context}

  규칙:
  - 반드시 위 목록에 있는 약만 사용할 수 있습니다.
  - 목록에 없는 약은 절대 사용할 수 없습니다.
  """


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


def create_expert_chatbot():
    prompt = ChatPromptTemplate.from_messages([
        ("system",         """
        당신은 약사입니다.

        먼저 사용자의 질문 의도를 반드시 분류하세요:

        1. [추천 요청]
        - 특정 약 이름 없이 증상만 있음.
        - 어떤 약이 좋은지 묻는 경우

        2. [약 효과 확인]
        - 특정 약 이름이 포함됨.
        - 이 약이 효과 있는지 묻는 경우


        공통으로 지켜야 할 규칙:
        - 답변에 사용하는 모든 약은 반드시 [약 ID]가 있는 context에서 선택합니다.
        - context에 없는 약은 절대 언급하지 않습니다.
        - 질문에 대해
        - [대표 약]을 중심으로 설명합니다.
        이때, 약의 성분, 제형, 사용 방법 등을 추가적으로 안내합니다.
        - 반드시 [유사 효능 약]에 있는 약 중에서 최대 2개를 추가 추천합니다.
        만약, [유사 효능 약]이 없다면 아무것도 추가 추천하지 않습니다.
        - 약 이름은 반드시 context에 있는 [약 이름] 값을 그대로 사용해야 합니다.
        - 임의로 약 이름을 생성하거나 변경 또는 유추하지 않습니다.
        - 주의 사항에는 부작용과 주의 사항을 모두 합해서 답변하기.

        주의:
        - 증상과 매칭할 수 있는 약이 없다면 추천하지 않습니다.

        [추천 요청일 경우 출력 형식]

        [대표 약]
        - 약 이름:
        - 효능:
        - 복용법:
        - 주의 사항:

        [추천 약]
        - 약 이름:
        - 이유:


        [약 효과 확인일 경우 출력 형식]

        [약 효과 판단]
        - 약 이름 :
        - 해당 증상에 적합한지 : (적합 / 부적합)
        - 이유

        [추가 추천]
        - (필요한 경우만 작성)
        - 약 이름:
        - 이유:

       context
       : {context}

       질문
       : {input}

       답변:
        """),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}")
    ])

    rag_chain = (
        RunnablePassthrough.assign(
            context = RunnableLambda(get_filtered_docs)
        )
        |prompt
        |llm
        |StrOutputParser()
    )


    store = {}

    def get_history(session_id):
        if session_id not in store:
            store[session_id] = InMemoryChatMessageHistory()
        return store[session_id]

    return RunnableWithMessageHistory(
        rag_chain,
        get_history,
        input_messages_key="input",
        history_messages_key="history"
    )


# 사용

chatbot = create_expert_chatbot()
config = {"configurable": {"session_id": "user1"}}

while True:
    user_input = input("\n👤 You: ").strip()
    if user_input.lower() in ["quit", "exit"]:
        break

    response = chatbot.invoke({"input": user_input}, config=config)
    print(f"\n🤖 AI: {response}")



👤 You: 눈의 피로가 있어. 어떤 약이 좋을까?

🤖 AI: [대표 약]
- 약 이름: 아이미루콘택트퓨어점안액
- 효능: 눈의 피로, 누액의 보조(눈의 건조), 하드콘택트렌즈 또는 소프트렌즈 착용시 불쾌감, 눈의 침침함(눈곱이 많을 때 등)에 사용합니다.
- 복용법: 1일 3~6회, 1회 1~3방울 점안합니다.
- 주의 사항: 이 약을 사용하기 전에 안약에 과민증 환자, 눈의 통증이 심한 환자, 녹내장 환자는 의사 또는 약사와 상의하십시오. 정해진 용법과 용량을 잘 지키십시오. 어린이에게 투여할 경우 보호자의 지도 감독하에 투여하십시오. 이 약은 점안용으로만 사용하십시오. 오염을 방지하기 위해 가능한 공동으로 사용하지 마십시오.

[추천 약]
- 약 이름: 아이미루40이엑스점안액
- 이유: 이 약도 눈의 피로를 완화하는데 효과가 있으며, 결막충혈 및 하드콘택트렌즈 착용 시 불쾌감에 사용됩니다.

👤 You: 두통은 없는데 기침은 나. 타이레놀이 효과가 있을까?

🤖 AI: [약 효과 판단]
- 약 이름 : 타이레놀콜드-에스정
- 해당 증상에 적합한지 : 부적합
- 이유: 타이레놀콜드-에스정은 감기의 여러 증상, 특히 두통, 발열, 관절통, 근육통 등을 완화하는데 사용되며, 기침에 대한 직접적인 효과는 없습니다. 또한, 두통이 없으므로 이 약의 사용이 적합하지 않습니다. 

[추가 추천]
- 해당 증상에 적합한 약이 없습니다.

👤 You: 눈의 피로가 있어. 게보린이 효과가 있을까?

🤖 AI: [약 효과 판단]
- 약 이름 : 게보린정
- 해당 증상에 적합한지 : 부적합
- 이유: 게보린정은 주로 두통, 치통, 발치 후 동통, 인후통, 귀의 통증 등 다양한 통증과 발열에 사용되는 진통제입니다. 눈의 피로에 대한 효과는 없으므로 적합하지 않습니다.

[추가 추천]
- 해당 증상에 적합한 약이 없습니다.

👤 You: quit


### 시나리오 1.
질문 : "눈의 피로가 있어. 어떤 약이 좋을까?"

답변:
🤖 AI: [대표 약]
- 약 이름: 아이미루콘택트퓨어점안액
- 효능: 이 약은 눈의 피로, 누액의 보조(눈의 건조), 하드콘택트렌즈 또는 소프트렌즈 착용시 불쾌감, 눈의 침침함(눈곱이 많을 때 등)에 사용합니다.
- 복용법: 1일 3~6회, 1회 1~3방울 점안합니다.
- 주의 사항: 이 약을 사용하기 전에 안약에 과민증 환자, 눈의 통증이 심한 환자, 녹내장 환자는 의사 또는 약사와 상의하십시오. 정해진 용법과 용량을 잘 지키십시오. 어린이에게 투여할 경우 보호자의 지도 감독하에 투여하십시오. 이 약은 점안용으로만 사용하십시오. 오염을 방지하기 위해 가능한 공동으로 사용하지 마십시오. 부작용으로는 눈의 충혈, 가려움, 부종이 있을 수 있으며, 하드콘택트렌즈를 착용하는 경우 눈곱 등의 분비물이 많거나 렌즈가 흐리게 되는 경우 즉시 사용을 중지하고 의사 또는 약사와 상담해야 합니다.

[추천 약]
- 약 이름: 아이미루40이엑스점안액
- 이유: 이 약도 눈의 피로와 하드콘택트렌즈 착용시 불쾌감을 완화하는 데 도움을 줄 수 있습니다.


### 시나리오 2.

질문 : "어린이용 해열제를 추천해줘."

답변: 🤖 AI: [대표 약]
- 약 이름: 어린이타이레놀산160밀리그램(아세트아미노펜)
- 효능: 이 약은 감기로 인한 발열 및 동통(통증), 두통, 신경통, 근육통, 월경통, 염좌통(삔 통증), 치통, 관절통, 류마티양 동통(통증)에 사용합니다.
- 복용법: 만 7~12세 소아는 1회 권장용량을 4~6시간마다 필요시 물 없이 혀에 직접 복용합니다. 이 약은 가능한 최단기간동안 최소 유효용량으로 복용하며, 1일 5회(75 mg/kg)를 초과하여 복용하지 않습니다. 몸무게를 아는 경우 몸무게에 따른 용량(10~15 mg/kg)으로 복용하는 것이 더 적절합니다. 연령대별 용량은 허가사항을 참고하십시오.
- 주의 사항: 이 약은 과다복용 시 간 손상을 초래할 수 있으므로 정해진 용법과 용량을 준수해야 합니다. 아세트아미노펜에 과민증이 있는 경우 사용을 피해야 하며, 다른 아세트아미노펜 성분이 포함된 약과 병용하지 않도록 주의해야 합니다. 부작용으로는 메스꺼움, 구토, 간 기능 이상 등이 있을 수 있습니다.

[추천 약]
- 약 이름: 테라플루나이트타임건조시럽
- 이유: 이 약은 감기의 여러 증상 완화에 효과적이며, 발열과 관련된 증상 완화에 도움을 줄 수 있습니다.


### 시나리오 3.
질문 : 두통은 있지만 기침은 안 나. 어떤 약이 좋을까?

답변 : 🤖 AI: [대표 약]
- 약 이름: 게보린릴랙스연질캡슐
- 효능: 이 약은 두통, 치통, 발치후 동통, 인후통, 귀의 통증, 관절통, 신경통, 요통, 근육통, 견통, 타박통, 골절통, 염좌통, 월경통, 외상통의 진통과 오한, 발열시의 해열에 사용합니다.
- 복용법: 만 15세 이상 및 성인은 1회 1캡슐씩, 1일 1~3회 공복(빈속)시를 피하여 복용합니다. 복용간격은 4시간 이상으로 합니다.
- 주의 사항: 이 약은 위장 장애가 있는 환자, 심부전 및 간질환 환자에게 주의하여 사용해야 하며, 장기간 복용을 피해야 합니다. 부작용으로는 위장장애, 두통, 어지러움, 알레르기 반응 등이 있을 수 있습니다.

[추천 약]
- 약 이름: 게보린정(수출명:돌로린정)
- 이유: 이 약도 두통 및 다양한 통증 완화에 효과적이며, 복용법이 간편하여 추천할 수 있습니다.

- 약 이름: 게보린소프트연질캡슐
- 이유: 이 약 역시 두통 완화에 효과적이며, 연질 캡슐 형태로 복용이 용이합니다.


### 시나리오 4.
질문 : 두통은 없는데 기침은 나. 타이레놀이 효과가 있을까?

답변: 🤖 AI: [약 효과 판단]
- 약 이름 : 타이레놀콜드-에스정
- 해당 증상에 적합한지 : 부적합
- 이유: 타이레놀콜드-에스정은 감기의 여러 증상, 특히 두통, 발열, 관절통, 근육통 등을 완화하는데 사용되며, 기침에 대한 직접적인 효과는 없습니다. 또한, 두통이 없으므로 이 약의 사용이 적합하지 않습니다.

[추가 추천]
- 해당 증상에 적합한 약이 없습니다.

### 시나리오 5.
질문 : 기침이 나. 게보린이 효과가 있을까?

답변 : 🤖 AI: [약 효과 판단]
- 약 이름 : 게보린정
- 해당 증상에 적합한지 : 부적합
- 이유: 게보린정은 주로 두통, 치통, 발치 후 동통, 인후통, 귀의 통증 등 다양한 통증과 발열에 사용되는 진통제입니다. 눈의 피로에 대한 효과는 없으므로 적합하지 않습니다.

[추가 추천]
- 해당 증상에 적합한 약이 없습니다